#### Limpeza dos dados da coletora

Os dados são a exportação da coletora, todos os pontos gravados de abril/2024 a outubro/2026.

- Limpeza e padronização dos dados para análise;
- Leitura com a codificação correta para o arquivo;
- Conversão de datas;
- Remoção de colunas com valores ausentes;
- Remoção de espaços em branco;
- Remoção de projetos de teste;
- Remoção de coordenadas impossíveis - pontos fora da região ou em 0,0;
- Remoção de linhas repetidas.

Critérios definidos junto à equipe de campo.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Configuração

# Garante a raiz do projeto como diretório de trabalho, tanto no Jupyter
# (kernel inicia em notebooks/) quanto no PyCharm (inicia na raiz do projeto).
raiz = Path.cwd()
if not (raiz / "dados").is_dir() and (raiz.parent / "dados").is_dir():
    os.chdir(raiz.parent)

ARQUIVO_BRUTO = "dados/dados_rtk_bruto.csv"
ARQUIVO_LIMPO = "dados/dados_rtk_limpo.csv"

# Projetos de teste conferidos com a equipe.
# "teste sinal vegetação" NÃO está na lista de propósito: é um dos temas da análise.
PROJETOS_TESTE = [
    "ijhh", "tes", "tesjte", "test", "test jair", "teste", "teste campo",
    "tets office sete", "tey", "zbsvhaj", "prioritário xxxxx", "Jão bbbbb",
]

# Janela da região atendida: extremos dos trabalhos reais + ~1° (~100 km) de folga
LAT_MIN, LAT_MAX = -23.0, -19.0
LON_MIN, LON_MAX = -52.0, -47.0

# 1. Leitura e datas

# "utf-8-sig" remove a marca BOM que a coletora grava no início do arquivo
dados = pd.read_csv(ARQUIVO_BRUTO, encoding="utf-8-sig", low_memory=False)
linhas_brutas, colunas_brutas = dados.shape
print(f"Arquivo bruto: {linhas_brutas} linhas, {colunas_brutas} colunas")

# Texto -> data e hora. "coerce" transforma em vazio (NaT) o que não for data
dados["Time"] = pd.to_datetime(dados["Time"], errors="coerce")
dados["STARTTIME"] = pd.to_datetime(dados["STARTTIME"], errors="coerce")
print(f"Linhas sem data (mantidas): {dados['Time'].isna().sum()}")

# 2. Colunas que têm sempre o mesmo valor

# nunique conta quantos valores diferentes cada coluna tem (dropna=False conta o vazio)
valores_diferentes = dados.nunique(dropna=False)
colunas_constantes = valores_diferentes[valores_diferentes == 1].index.tolist()
dados = dados.drop(columns=colunas_constantes)
print(f"\n2. Colunas constantes removidas ({len(colunas_constantes)}): {colunas_constantes}")

# 3. Espaços sobrando nos nomes (antes da etapa 4, para os nomes compararem certo)

for coluna in ["Projeto", "PointName"]:
    antes = dados[coluna].copy()
    # strip tira espaços das pontas; replace troca espaços repetidos por um só
    dados[coluna] = dados[coluna].str.strip().str.replace(r"\s+", " ", regex=True)
    print(f"3. {coluna}: {(antes != dados[coluna]).sum()} nomes corrigidos")

# 4. Projetos de teste

eh_teste = dados["Projeto"].isin(PROJETOS_TESTE)
removidas_teste = eh_teste.sum()
dados = dados[~eh_teste]   # o "~" inverte: fica o que NÃO é teste
print(f"4. Linhas de projetos de teste removidas: {removidas_teste}")

# 5. Coordenadas descartadas

dentro_da_regiao = (dados["Lat84"].between(LAT_MIN, LAT_MAX)
                    & dados["Lon84"].between(LON_MIN, LON_MAX))
fora = dados[~dentro_da_regiao]
dados = dados[dentro_da_regiao]
print(f"5. Linhas com coordenada fora da região removidas: {len(fora)}")
print(fora[["Projeto", "PointName", "PointType", "Lat84", "Lon84"]].to_string())

# 6. Linhas 100% repetidas (todas as colunas iguais)

repetidas = dados.duplicated()
dados = dados[~repetidas]
print(f"\n6. Linhas 100% repetidas removidas: {repetidas.sum()}")

# Resultado

dados.to_csv(ARQUIVO_LIMPO, index=False, encoding="utf-8-sig")

print(f"\nArquivo limpo: {len(dados)} linhas, {dados.shape[1]} colunas, "
      f"{dados['Projeto'].nunique()} projetos")
print(f"Removido no total: {linhas_brutas - len(dados)} linhas e "
      f"{colunas_brutas - dados.shape[1]} colunas")
print(f"Salvo em: {ARQUIVO_LIMPO}")

Arquivo bruto: 46792 linhas, 56 colunas
Linhas sem data (mantidas): 3

2. Colunas constantes removidas (7): ['RefBaseAntP', 'RefBaseAntC', 'RefBaseAntO', 'CS_Distance', 'CS_x', 'CS_y', 'CS_z']
3. Projeto: 488 nomes corrigidos
3. PointName: 1893 nomes corrigidos


4. Linhas de projetos de teste removidas: 290
5. Linhas com coordenada fora da região removidas: 30
                    Projeto      PointName  PointType      Lat84      Lon84
107         20231227_122107            PT1          0 -22.686523 -43.751848
108         20231227_122107         Base_0          6 -22.686523 -43.751848
109         20231227_122107            PT2          0 -22.686532 -43.751889
110         20231227_122107            PT3          0 -22.686520 -43.751850
111         20231227_122107         Base_1          6 -22.686514 -43.751849
112         20231227_122107         Base_2          6 -22.686520 -43.751850
113         20231227_122107         Base_3          6 -22.686514 -43.751849
114         20231227_122107         Base_4          6 -22.686520 -43.751850
115         20231227_122107            PT4          0 -22.686520 -43.751850
8832   cerca laranja noasaa             bb          2 -13.992128  -2.563602
16475        estação pavani           pt 2          2 -82.706173


Arquivo limpo: 46472 linhas, 49 colunas, 523 projetos
Removido no total: 320 linhas e 7 colunas
Salvo em: dados/dados_rtk_limpo.csv
